In [4]:
import time
import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import os
import re
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class EGATRealTimeScraper:
    def __init__(self, url="https://www.sothailand.com/sysgen/egat/", output_file="egat_power_data.csv"):
        self.url = url
        self.output_file = output_file
        self.driver = None
        self._initialize_driver()
        
    def _initialize_driver(self):
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--log-level=0")
        chrome_options.set_capability('goog:loggingPrefs', {'browser': 'ALL'})
        
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)

    def extract_data_from_console(self):
        if not self.driver:
            return None

        logs = self.driver.get_log('browser')
        found_data = None

        for log_entry in logs:
            message = log_entry.get('message', '')
            if 'updateMessageArea:' in message:
                match = re.search(r'updateMessageArea:\s+(\d+)\s*,\s*(\d+:\d+)\s*,\s*([\d,]+\.\d+)\s*,\s*(\d+\.\d+)', message)
                if match:
                    display_date_id = match.group(1).strip()
                    display_time = match.group(2).strip()
                    current_value_mw = float(match.group(3).replace(',', '').strip())
                    temperature_c = float(match.group(4).strip())

                    found_data = {
                        'scrape_time': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                        'display_date_id': display_date_id,
                        'display_time': display_time,
                        'current_value_MW': current_value_mw,
                        'temperature_C': temperature_c
                    }
        return found_data

    def save_to_csv(self, new_data):
        if not new_data:
            return None

        df_new = pd.DataFrame([new_data])
        
        if os.path.exists(self.output_file):
            df_new.to_csv(self.output_file, mode='a', header=False, index=False, encoding='utf-8')
        else:
            df_new.to_csv(self.output_file, index=False, encoding='utf-8')
            
        return df_new

    def scrape_once(self):
        if self.driver is None:
            self._initialize_driver()
            
        self.driver.get(self.url)
        time.sleep(5)
        
        data = self.extract_data_from_console()
        
        if data:
            self.save_to_csv(data)
        
        return data

    def scrape_continuously(self, interval_minutes=5):
        if interval_minutes <= 0:
            return
            
        while True:
            start_time = time.time()
            self.scrape_once()
            
            elapsed_time = time.time() - start_time
            sleep_time = max(0, (interval_minutes * 60) - elapsed_time)
                
            if sleep_time > 0:
                time.sleep(sleep_time)

    def close(self):
        if self.driver:
            self.driver.quit()
            self.driver = None

if __name__ == "__main__":
    scraper = EGATRealTimeScraper(output_file="egat_realtime_power.csv")
    scraper.scrape_continuously(interval_minutes=5)
    scraper.close()

2025-05-15 15:04:25,586 - INFO - ====== WebDriver manager ======
2025-05-15 15:04:26,683 - INFO - Get LATEST chromedriver version for google-chrome
2025-05-15 15:04:26,907 - INFO - Get LATEST chromedriver version for google-chrome
2025-05-15 15:04:27,063 - INFO - Driver [C:\Users\kongl\.wdm\drivers\chromedriver\win64\136.0.7103.94\chromedriver-win32/chromedriver.exe] found in cache


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=136.0.7103.93)
Stacktrace:
	GetHandleVerifier [0x0103FC83+61635]
	GetHandleVerifier [0x0103FCC4+61700]
	(No symbol) [0x00E605D3]
	(No symbol) [0x00E4FE20]
	(No symbol) [0x00E6DD1F]
	(No symbol) [0x00ED3E8C]
	(No symbol) [0x00EEDF19]
	(No symbol) [0x00ECD096]
	(No symbol) [0x00E9C840]
	(No symbol) [0x00E9D6A4]
	GetHandleVerifier [0x012C45A3+2701795]
	GetHandleVerifier [0x012BFD26+2683238]
	GetHandleVerifier [0x012DAA6E+2793134]
	GetHandleVerifier [0x01056945+155013]
	GetHandleVerifier [0x0105D02D+181357]
	GetHandleVerifier [0x010474D8+92440]
	GetHandleVerifier [0x01047680+92864]
	GetHandleVerifier [0x01032070+5296]
	BaseThreadInitThunk [0x76625D49+25]
	RtlInitializeExceptionChain [0x77C4CFFB+107]
	RtlGetAppContainerNamedObjectPath [0x77C4CF81+561]
